# Triple-Point Shock Interaction (2D) -- Equal Mass

This notebook runs the triple-point shock-interaction benchmark: three constant states in a 14x6 box (a dense/high-pressure corner, a light/low-pressure corner, and a dense/low-pressure corner) that collide and roll up into a Kelvin-Helmholtz-unstable shear layer. `triplePointCase.configureScheme` sets that box explicitly (`domain.min`/`.max`), not the symmetric cube `buildDomainDescription` would give it.

This variant (`equalMass=True`) samples the light region `sqrt(8)` times coarser than the two dense regions, so every particle in the domain carries the same mass instead of the same spacing -- the region most prone to under-resolving mass at a density jump gets compensated for it. Compare against `triplePoint_equalSpacing.ipynb`.

One `Case` covers both sampling strategies -- `equalMass=True` (this notebook) samples the light region `sqrt(8)` times coarser so every particle carries the same mass, `equalMass=False` (`triplePoint_equalSpacing.ipynb`) uses one particle spacing everywhere -- selected by `spec.params['equalMass']` the way `sedovCase` selects dimension by `spec.dim`. `triplePointCase` also has a `timestep` hook (`compressibleTimestep`), so the loop below is a `while t < tLimit`, the same shape as `03-kidder-isentropic-compression.ipynb`'s (no `postStep` here, unlike Kidder).

This is a `particlePlot` (2D field view) case: plotting calls `buildFieldPlotter`/`refreshFieldPlotter` directly on `TRIPLE_POINT_FIELDS` (exported from `warpSPH.cases.triplePoint`) rather than going through `triplePointCase.setupPlot`/`updatePlot`, which do not live-update reliably inside a Jupyter cell in this environment -- the same reasoning `08-hydrostatic.ipynb` (the pilot for this shape) explains in full.

Precision note: switching between single and double precision is controlled in the import/configuration cell below. Because precision is set when core modules/kernels are initialized, any precision change requires a kernel restart before re-running the notebook.

![](outputs/14-Triple_Point_equal_mass.gif)


In [ ]:
%matplotlib inline
from warpSPHBootstrap import bootstrap
rt = bootstrap(precision='float32', verbose=True)

from warpSPH import *
from warpSPH.cases.triplePoint import triplePointCase, TRIPLE_POINT_FIELDS
from warpSPH.cases.plotting import buildFieldPlotter, refreshFieldPlotter
from warpSPH.runner import CaseSpec, buildContext, encodeFrames
from warpSPH.io import createOutFile, prepExport, writeInitialData, writeFrame

import os
import torch
from tqdm.autonotebook import tqdm


In [ ]:
# Every knob you'd otherwise reach for as a `--flag` on
# `triplePoint_equalMass.py`, made explicit and editable here.
# `triplePointCase.defaults`/`.params` are the same values the CLI script
# starts from -- anything not overridden below just keeps its case default.
spec = CaseSpec(caseName=triplePointCase.name, scheme=triplePointCase.scheme,
                params=dict(triplePointCase.params)) \
    .merged(**triplePointCase.defaults)

spec = spec.merged(
    # --- discretisation ------------------------------------------------
    nx=256,
    dim=2,

    # --- time stepping ---------------------------------------------------
    tLimit=10.0,
    # No `dt` here -- `triplePointCase.timestep` (`compressibleTimestep`)
    # re-picks it every step, so the loop below is `while t < tLimit`.

    # --- output --------------------------------------------------------------
    caseName='14-triplePointEqualMass',
    plot=True, show=True, plotInterval=25,
    store=False,
    # vispy's default 'notebook' mode (the jupyter_rfb widget) needs a live
    # ipywidgets comm channel to the browser, which doesn't come up over this
    # SSH-remote VSCode connection even though the kernel itself renders fine.
    # 'image' mode instead renders fully offscreen on a headless EGL GL
    # context (no window, no comm) and pushes frames as plain Jupyter image
    # updates -- still GPU-accelerated, so it stays fast for the ~3M
    # particles here where matplotlib would not.
    plotBackendOptions={'jupyter_backend': 'image', 'app_backend': 'egl'},

    # --- triple-point's own knobs (sampling strategy + three states) ---------
    params=dict(
        gamma=1.4,
        equalMass=True,
        splitX=1.0, splitY=1.5,
        rho_I=1.0, p_I=1.0,
        rho_II=0.125, p_II=0.1,
        rho_III=1.0, p_III=0.1,
    ),
)
spec


In [ ]:
# Initial-condition generation: explicit, using the real case code
# (`triplePointCase.buildSystem` -> `sampleTriplePointEqualMass`/
# `sampleTriplePointEqualResolution`, picked by `equalMass`), not re-derived
# here. `configureScheme` is the case's own -- it sets the 14x6 box before
# deferring to the shared compressible setup, see the intro cell.
ctx = buildContext(triplePointCase, spec)
triplePointCase.configureScheme(ctx)
system = triplePointCase.buildSystem(ctx)
runningState = system.initializeNewState()


In [ ]:
# Export/plot setup via the same generic hooks `warpSPH.runner.run()` uses
# internally -- nothing here is re-derived, only called explicitly.
ctx.exportPath = prepExport(spec.caseName, ctx.config, ctx.schemeConfig, ctx.scheme, ctx.exportFunction)
spec.save(os.path.join(ctx.exportPath, 'caseSpec.json'))
print(f'exporting to {ctx.exportPath}')

# Direct buildFieldPlotter(TRIPLE_POINT_FIELDS), not triplePointCase.setupPlot
# -- see the intro cell for why.
plotter = None
if spec.plot:
    ctx.imagePath = os.path.join(ctx.exportPath, 'images')
    os.makedirs(ctx.imagePath, exist_ok=True)
    plotter = buildFieldPlotter(ctx, runningState, TRIPLE_POINT_FIELDS, figsize=(12, 6))

outFile = None
groups = None
if spec.store and spec.storeMode == 'trajectory':
    extraData = triplePointCase.extraData(ctx, runningState)
    outFile = createOutFile(ctx.exportPath)
    groups = writeInitialData(ctx.exportPath, outFile, ctx.scheme, ctx.config, ctx.schemeConfig,
                              spec, runningState, extraData=extraData, extraFields=triplePointCase.extraFields)


In [ ]:
import warp as wp
warp_memory = wp.get_mempool_used_mem_current() / (1024 ** 2)  # in MB
print(f"Initial Warp memory usage: {warp_memory:.2f} MB")

In [ ]:
# The step loop, visible and editable. This is the same call
# `warpSPH.runner.runner._run` makes internally, unrolled here so a
# perturbation or an extra diagnostic can be injected directly around it.
# `triplePointCase.timestep` is what makes this a `while t < tLimit` loop
# rather than a fixed `range(nSteps)` -- see the intro cell.
dt0 = ctx.config.dt if isinstance(ctx.config.dt, float) else ctx.config.dt.cpu().item()
storeSteps = max(1, int(spec.exportInterval / dt0)) if spec.storeMode == 'trajectory' \
    else max(1, spec.storeInterval)

trajectory = []
tq = tqdm(total=1000, leave=True)
i = 0
t = 0.0
# torch.cuda.memory._record_memory_history(max_entries=100000)

while t < spec.tLimit:
    begin = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    begin.record()
    # <-- hook point ---------------------------------------------------------
    stepResult = ctx.integrator.function(
        state=runningState, f=ctx.stepFunction, dt=ctx.config.dt,
        config=ctx.config, schemeConfig=ctx.schemeConfig, verbose=False,
    )
    runningState = stepResult.state
    ctx.config.dt = triplePointCase.timestep(ctx, runningState)
    # -------------------------------------------------------------------------
    end.record()

    t = runningState.t.item() if torch.is_tensor(runningState.t) else runningState.t
    row = triplePointCase.diagnostics(ctx, runningState)
    # trajectory.append(dict(row, step=i, t=t))

    final = t >= spec.tLimit
    if plotter is not None and (i % spec.plotInterval == 0 or final):
        refreshFieldPlotter(ctx, runningState, plotter, TRIPLE_POINT_FIELDS, step=i)

    if outFile is not None and (i % storeSteps == 0 or final):
        writeFrame(groups, i, stepResult.state, stepResult.stages, config=ctx.config,
                  schemeConfig=ctx.schemeConfig, uniqueParticles=True, writeStages=False,
                  extraFields=triplePointCase.extraFields)

    current_memory_allocated = torch.cuda.memory_allocated() / (1024 ** 2)  # in MB
    current_memory_reserved = torch.cuda.memory_reserved() / (1024 ** 2)

    warp_memory = wp.get_mempool_used_mem_current() / (1024 ** 2)  # in MB

    if i % 10 == 0:
        torch.cuda.empty_cache()


    tq.n = min(1000, int(t / spec.tLimit * 1000))
    tq_text = f"t: {t:.4f}, " + ", ".join(f"{k}: {v:.4f}" for k, v in row.items())
    tq_text += f", Mem Alloc: {current_memory_allocated:.2f} MB, Mem Reserved: {current_memory_reserved:.2f} MB Warp Mem: {warp_memory:.2f} MB iterTime: {begin.elapsed_time(end):.2f} ms"
    tq.set_description(tq_text)
    tq.refresh()

    i += 1
    # if i > 32:
    #     break
tq.close()
# torch.cuda.memory._dump_snapshot("memory_snapshot.json")

In [ ]:
if outFile is not None:
    outFile.close()

if spec.plot:
    encodeFrames(ctx.imagePath, ctx.exportPath)
